# 08 · Geometría de cámara: calibración, homografías y multicámara

La oferta valora *"homografías, calibración de cámaras y sistemas multicámara"* y pide *"integración de
información visual, espacial y temporal"*. En deporte esto es lo que convierte cajas en píxeles en
**posiciones en metros sobre el campo** → distancias, velocidades, mapas de calor, formaciones.

1. Modelo pinhole: intrínsecos y extrínsecos ✍️
2. Distorsión
3. Calibración con tablero de ajedrez (sintético, con K conocida para comprobar)
4. Homografía: qué es, cuándo vale, DLT desde cero ✍️, RANSAC
5. De píxeles a metros sobre el campo: vista cenital, qué punto de la caja proyectar
6. Cámaras en movimiento y multicámara

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=3)

## 1. El modelo pinhole

Un punto 3D del mundo $X_w$ se proyecta al píxel $(u, v)$:

$$ s \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \,[R \mid t]\, \begin{bmatrix} X \\ Y \\ Z \\ 1 \end{bmatrix},
\qquad K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix} $$

- **Extrínsecos** $[R \mid t]$: pose de la cámara (rotación 3×3 + traslación) → pasan del sistema del mundo al
  de la cámara: $X_c = R X_w + t$. 6 grados de libertad.
- **Intrínsecos** $K$: propios de la cámara+óptica. $f_x, f_y$ = focal en píxeles ($f_{mm} \cdot$ píxeles/mm);
  $(c_x, c_y)$ = punto principal (≈ centro de la imagen). Cambian si cambias el **zoom**.
- $s = Z_c$ (profundidad): la división por $s$ es la **perspectiva** (lo lejano se ve pequeño). Al dividir se
  pierde la profundidad → **con una sola cámara no puedes recuperar 3D**… salvo que sepas que el punto está en
  un plano conocido (el suelo) → homografía.

✍️ **EJERCICIO**: implementa `project(Xw, K, R, t)` → píxeles `(N, 2)` y compáralo con `cv2.projectPoints`.

In [ ]:
def project(Xw, K, R, t):
    # TU CÓDIGO AQUÍ
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def project(Xw, K, R, t):
    Xc = Xw @ R.T + t.reshape(1, 3)          # mundo → cámara
    uvw = Xc @ K.T                            # cámara → plano imagen (homogéneo)
    return uvw[:, :2] / uvw[:, 2:3]           # división perspectiva

In [ ]:
W, H = 1280, 720
K = np.array([[1000., 0, W / 2], [0, 1000., H / 2], [0, 0, 1]])
rvec = np.array([0.3, -0.2, 0.05]); R, _ = cv2.Rodrigues(rvec)    # rotación como vector de Rodrigues (eje·ángulo)
t = np.array([-0.5, 0.2, 6.0])
Xw = np.random.default_rng(0).uniform(-1, 1, (10, 3))
mine = project(Xw, K, R, t)
ref, _ = cv2.projectPoints(Xw, rvec, t, K, None)
print("error máx vs OpenCV:", np.abs(mine - ref.reshape(-1, 2)).max())

## 2. Distorsión de la lente

Las lentes reales (sobre todo gran angular / ojo de pez) curvan las rectas. Modelo de OpenCV:
- **Radial**: $x_d = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$ → efecto barril ($k_1<0$) o cojín ($k_1>0$).
- **Tangencial** ($p_1, p_2$): lente no paralela al sensor.
- `cv2.undistort(img, K, dist)` corrige la imagen; `cv2.undistortPoints` corrige solo puntos (más barato: en
  tracking basta con corregir los pies de las cajas antes de aplicar la homografía).
- Con distorsión fuerte **la homografía no es válida** sobre la imagen sin corregir: las líneas del campo no son
  rectas. Primero *undistort*, luego homografía.

In [ ]:
dist = np.array([-0.35, 0.12, 0, 0, 0])               # k1, k2, p1, p2, k3 (barril)
grid = np.stack(np.meshgrid(np.linspace(-1.5, 1.5, 15), np.linspace(-0.9, 0.9, 9)), -1).reshape(-1, 2)
pts3 = np.c_[grid, np.zeros(len(grid))]
p_ideal, _ = cv2.projectPoints(pts3, np.zeros(3), np.array([0, 0, 2.0]), K, None)
p_dist, _ = cv2.projectPoints(pts3, np.zeros(3), np.array([0, 0, 2.0]), K, dist)
p_fix = cv2.undistortPoints(p_dist, K, dist, P=K)
plt.figure(figsize=(10, 5))
plt.scatter(*p_ideal.reshape(-1, 2).T, s=8, label="ideal"); plt.scatter(*p_dist.reshape(-1, 2).T, s=8, label="con distorsión")
plt.scatter(*p_fix.reshape(-1, 2).T, s=30, facecolors="none", edgecolors="k", label="corregidos")
plt.gca().invert_yaxis(); plt.legend(); plt.title("distorsión radial (barril) y corrección");

## 3. Calibración con tablero de ajedrez

Procedimiento (Zhang, 2000) que implementa `cv2.calibrateCamera`:
1. Imprimir un tablero de dimensiones conocidas y fotografiarlo desde 15–30 poses distintas (inclinado, en las
   esquinas de la imagen, a varias distancias).
2. `findChessboardCorners` + `cornerSubPix` en cada imagen → píxeles de las esquinas interiores.
3. Puntos 3D del tablero en su propio sistema: $(i \cdot \text{tam}, j \cdot \text{tam}, 0)$.
4. `calibrateCamera` estima $K$, distorsión y la pose de cada imagen minimizando el **error de reproyección**
   (distancia entre esquinas detectadas y reproyectadas). < 0.5 px es una buena calibración.

Aquí **generamos** las fotos: renderizamos un tablero con una $K$ conocida en poses aleatorias (con
`warpPerspective`, porque un plano se proyecta con una homografía $H = K[r_1\; r_2\; t]$) y comprobamos que la
calibración recupera esa $K$.

In [ ]:
cols, rows, sq = 9, 6, 60                  # esquinas interiores y tamaño del cuadro en px de la textura
board = np.kron((np.indices((rows + 1, cols + 1)).sum(0) % 2) * 255, np.ones((sq, sq))).astype(np.uint8)
board = cv2.copyMakeBorder(board, sq, sq, sq, sq, cv2.BORDER_CONSTANT, value=255)
square_m = 0.025                           # 2.5 cm por cuadro en el "mundo"
px2m = square_m / sq

def render_board(K, rvec, t):
    R, _ = cv2.Rodrigues(rvec)
    # punto de textura (u,v) → mundo (u·px2m − borde, v·px2m − borde, 0) → imagen
    S = np.array([[px2m, 0, -sq * px2m], [0, px2m, -sq * px2m], [0, 0, 1]])
    Hb = K @ np.c_[R[:, 0], R[:, 1], t] @ S
    img = cv2.warpPerspective(board, Hb, (W, H), borderValue=128)
    return cv2.GaussianBlur(img, (3, 3), 0)

rng = np.random.default_rng(1)
objp = np.zeros((rows * cols, 3), np.float32)
objp[:, :2] = (np.mgrid[1:cols + 1, 1:rows + 1].T.reshape(-1, 2)) * square_m   # esquinas interiores en metros
obj_pts, img_pts, shown = [], [], []
for k in range(25):
    rv = rng.uniform(-0.5, 0.5, 3)
    tv = np.array([rng.uniform(-0.25, 0.05), rng.uniform(-0.15, 0.0), rng.uniform(0.45, 0.8)])
    img = render_board(K, rv, tv)
    found, corners = cv2.findChessboardCorners(img, (cols, rows))
    if not found:
        continue
    corners = cv2.cornerSubPix(img, corners, (5, 5), (-1, -1),
                               (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3))
    obj_pts.append(objp); img_pts.append(corners)
    if len(shown) < 3: shown.append(cv2.drawChessboardCorners(cv2.cvtColor(img, cv2.COLOR_GRAY2BGR), (cols, rows), corners, found))
print(f"tablero encontrado en {len(img_pts)}/25 imágenes")
fig, axs = plt.subplots(1, len(shown), figsize=(18, 4))
for a, s in zip(np.atleast_1d(axs), shown): a.imshow(s[:, :, ::-1]); a.axis("off")
plt.show()

In [ ]:
rms, K_est, dist_est, rvecs, tvecs = cv2.calibrateCamera(obj_pts, img_pts, (W, H), None, None)
print(f"error de reproyección RMS: {rms:.3f} px")
print("K real:\n", K, "\nK estimada:\n", K_est)
print("distorsión estimada (≈0, porque no la simulamos):", dist_est.ravel())

## 4. Homografía

Una **homografía** es una transformación proyectiva 2D → 2D: $\tilde p' \sim H \tilde p$, con $H$ 3×3 definida
salvo escala → **8 grados de libertad** → necesita **≥ 4 correspondencias** de puntos (cada una da 2 ecuaciones),
sin 3 colineales.

**¿Cuándo es exacta?**
1. Puntos que están en un **plano** (el suelo / el campo) vistos por una cámara: $H = K [r_1\; r_2\; t]$
   (las columnas de $R$ para $X$ e $Y$ cuando $Z = 0$).
2. Dos imágenes de una cámara que solo **rota** (sin trasladarse): $H = K R K^{-1}$ → panorámicas, y la cámara
   de TV que hace paneo/tilt/zoom desde un punto fijo.

**No vale** para puntos fuera del plano: la cabeza de un jugador (a 1.8 m) proyectada con la homografía del
suelo cae en el sitio equivocado → **proyecta siempre el punto de contacto con el suelo**: el centro del borde
inferior de la caja `((x1+x2)/2, y2)` o, mejor, el punto medio de los tobillos (pose).

### DLT (Direct Linear Transform)
De $\tilde p' \times H \tilde p = 0$ salen 2 ecuaciones lineales en los 9 elementos de $H$ por correspondencia:

$$ \begin{bmatrix} -x & -y & -1 & 0 & 0 & 0 & x u & y u & u \\ 0 & 0 & 0 & -x & -y & -1 & x v & y v & v \end{bmatrix} h = 0 $$

Apilando N ≥ 4 correspondencias: $A h = 0$, $A \in \mathbb{R}^{2N \times 9}$. La solución (mínimos cuadrados con
$\|h\| = 1$) es el **vector singular derecho de menor valor singular** de $A$ (última fila de $V^\top$ en la SVD).
Para estabilidad numérica, **normalizar** los puntos antes (Hartley: centrar y escalar a distancia media √2).

✍️ **EJERCICIO**: implementa `dlt_homography(src, dst)` (con normalización de Hartley).

In [ ]:
def dlt_homography(src, dst):
    # TU CÓDIGO AQUÍ
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def _normalize(p):
    c = p.mean(0)
    s = np.sqrt(2) / np.mean(np.linalg.norm(p - c, axis=1))
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
    ph = np.c_[p, np.ones(len(p))] @ T.T
    return ph[:, :2], T

def dlt_homography(src, dst):
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    s, Ts = _normalize(src); d, Td = _normalize(dst)
    A = []
    for (x, y), (u, v) in zip(s, d):
        A.append([-x, -y, -1, 0, 0, 0, x * u, y * u, u])
        A.append([0, 0, 0, -x, -y, -1, x * v, y * v, v])
    _, _, Vt = np.linalg.svd(np.array(A))
    Hn = Vt[-1].reshape(3, 3)
    Hm = np.linalg.inv(Td) @ Hn @ Ts          # deshacer la normalización
    return Hm / Hm[2, 2]

def apply_h(Hm, pts):
    p = np.c_[pts, np.ones(len(pts))] @ Hm.T
    return p[:, :2] / p[:, 2:3]

## 5. De la imagen al campo: un caso de fútbol

Construimos un **campo de fútbol de 105 × 68 m** visto por una cámara de retransmisión (desde la grada,
inclinada) con una $K$ y una pose conocidas. Así sabemos la verdad y medimos errores.

In [ ]:
FIELD_W, FIELD_H = 105.0, 68.0

def field_lines():
    """Segmentos de las líneas principales del campo, en metros (origen en una esquina)."""
    L = [((0, 0), (105, 0)), ((105, 0), (105, 68)), ((105, 68), (0, 68)), ((0, 68), (0, 0)), ((52.5, 0), (52.5, 68))]
    for x0, s in [(0, 1), (105, -1)]:
        L += [((x0, 13.84), (x0 + s * 16.5, 13.84)), ((x0 + s * 16.5, 13.84), (x0 + s * 16.5, 54.16)),
              ((x0 + s * 16.5, 54.16), (x0, 54.16)),
              ((x0, 24.84), (x0 + s * 5.5, 24.84)), ((x0 + s * 5.5, 24.84), (x0 + s * 5.5, 43.16)), ((x0 + s * 5.5, 43.16), (x0, 43.16))]
    ang = np.linspace(0, 2 * np.pi, 60)
    circ = np.c_[52.5 + 9.15 * np.cos(ang), 34 + 9.15 * np.sin(ang)]
    L += [(tuple(a), tuple(b)) for a, b in zip(circ[:-1], circ[1:])]
    return L

# Cámara de retransmisión: en la grada, a 30 m del lateral, 15 m de altura, mirando al centro del campo
Kc = np.array([[1400., 0, 640], [0, 1400., 360], [0, 0, 1]])
cam_pos = np.array([52.5, -30.0, 15.0])
target = np.array([52.5, 34.0, 0.0])
z = target - cam_pos; z /= np.linalg.norm(z)                 # eje óptico
x = np.cross(z, [0, 0, 1.0]); x /= np.linalg.norm(x)           # derecha de la imagen
y = np.cross(z, x)                                             # abajo de la imagen
Rc = np.stack([x, y, z]); tc = -Rc @ cam_pos
H_true = Kc @ np.c_[Rc[:, 0], Rc[:, 1], tc]; H_true /= H_true[2, 2]   # campo (Z=0, metros) → imagen

img = np.full((720, 1280, 3), (60, 140, 60), np.uint8)
for a, b in field_lines():
    pa, pb = apply_h(H_true, np.array([a, b], float)).astype(int)
    cv2.line(img, tuple(pa), tuple(pb), (255, 255, 255), 2)

# Jugadores: posición en el campo (m) → caja en la imagen (la altura en px sale de proyectar la cabeza a 1.8 m)
rng = np.random.default_rng(3)
players_m = np.c_[rng.uniform(20, 85, 12), rng.uniform(8, 60, 12)]
boxes = []
for X, Y in players_m:
    foot = project(np.array([[X, Y, 0.0]]), Kc, Rc, tc)[0]
    head = project(np.array([[X, Y, 1.8]]), Kc, Rc, tc)[0]
    h = foot[1] - head[1]; w = 0.4 * h
    boxes.append([foot[0] - w / 2, head[1], foot[0] + w / 2, foot[1]])
    cv2.rectangle(img, (int(foot[0] - w / 2), int(head[1])), (int(foot[0] + w / 2), int(foot[1])), (0, 0, 255), 2)
boxes = np.array(boxes)
plt.figure(figsize=(14, 8)); plt.imshow(img[:, :, ::-1]); plt.title("vista de la cámara (sintética)"); plt.axis("off");

### Estimar H a partir de puntos de referencia
En la vida real se marcan ≥ 4 puntos conocidos del campo en la imagen (esquinas del área, intersección de la
línea central con las bandas…), a mano una vez por cámara fija, o con un **modelo de detección de keypoints del
campo** en cada frame si la cámara se mueve. Simulamos el clic con un poco de ruido (±2 px) y algún error
grosero (*outlier*) para usar **RANSAC**.

In [ ]:
ref_field = np.array([[0, 0], [105, 0], [52.5, 0], [52.5, 68], [16.5, 13.84], [16.5, 54.16], [88.5, 13.84],
                      [88.5, 54.16], [0, 13.84], [105, 13.84], [52.5, 24.85], [52.5, 43.15]], float)
ref_img = apply_h(H_true, ref_field) + rng.normal(0, 2, (len(ref_field), 2))
ref_img[3] += [60, -40]                                       # un clic erróneo
H_dlt = dlt_homography(ref_img, ref_field)                    # imagen → campo
H_cv, inl = cv2.findHomography(ref_img, ref_field, cv2.RANSAC, ransacReprojThreshold=5.0)
H_inv_true = np.linalg.inv(H_true)

foot_px = np.c_[(boxes[:, 0] + boxes[:, 2]) / 2, boxes[:, 3]]
center_px = np.c_[(boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2]
good = inl.ravel() == 1
H_dlt_in = dlt_homography(ref_img[good], ref_field[good])
for name, Hm in [("H real", H_inv_true), ("DLT (con outlier)", H_dlt), ("RANSAC OpenCV", H_cv), ("mi DLT sin outlier", H_dlt_in)]:
    err = np.linalg.norm(apply_h(Hm, foot_px) - players_m, axis=1)
    print(f"{name:18s} error medio en el campo: {err.mean():5.2f} m  (máx {err.max():.2f} m)")
print("inliers RANSAC:", inl.ravel())
err_c = np.linalg.norm(apply_h(H_cv, center_px) - players_m, axis=1)
print(f"\nproyectando el CENTRO de la caja en vez del pie: error medio {err_c.mean():.2f} m  ← ¡por eso se usa el pie!")

In [ ]:
# Minimapa (vista cenital) con las posiciones estimadas
def draw_pitch(ax):
    for a, b in field_lines():
        ax.plot([a[0], b[0]], [a[1], b[1]], "w-", lw=1)
    ax.set_facecolor((0.23, 0.55, 0.23)); ax.set_xlim(-3, 108); ax.set_ylim(71, -3); ax.set_aspect("equal")

fig, ax = plt.subplots(figsize=(10, 6.5)); draw_pitch(ax)
est = apply_h(H_cv, foot_px)
ax.scatter(*players_m.T, c="yellow", s=60, label="real")
ax.scatter(*est.T, c="red", marker="x", s=60, label="estimado (pie + homografía)")
ax.legend(loc="lower right"); ax.set_title("posiciones en metros");

In [ ]:
# Warp de la imagen completa a vista cenital (10 px por metro)
S = np.diag([10, 10, 1.0])
top = cv2.warpPerspective(img, S @ H_cv, (1050, 680))
plt.figure(figsize=(12, 8)); plt.imshow(top[:, :, ::-1]); plt.title("imagen reproyectada al plano del campo (los jugadores se 'estiran': no están en el plano)"); plt.axis("off");

Fíjate en la última imagen: el césped y las líneas quedan perfectos, pero los jugadores aparecen **tumbados y
estirados** hacia fuera de la cámara: la homografía solo es correcta para lo que está en el plano.

### Una vez en metros…
- **Distancia recorrida** y **velocidad** (m/s, km/h) → con suavizado (notebook 09), porque el ruido de la caja
  se amplifica al derivar.
- **Asociación en coordenadas del campo**: en vista de retransmisión, dos jugadores pueden solaparse mucho en
  la imagen aunque estén a 3 m en el campo; asociar con distancias en metros + Kalman en metros reduce switches.
- Mapas de calor, distancias entre líneas, fuera de juego, presión, etc.

**Error de la homografía según la distancia**: un píxel de error lejos de la cámara equivale a muchos más metros
que cerca (la perspectiva comprime). Por eso las posiciones del fondo del campo son menos precisas.

In [ ]:
for Y in [5, 34, 63]:
    p = apply_h(H_true, np.array([[52.5, Y]]))
    q = apply_h(H_inv_true, p + [0, 1])            # 1 px más abajo en la imagen
    print(f"jugador a Y={Y:2d} m del lateral cercano: 1 px de error vertical ≈ {abs(q[0, 1] - Y) * 100:.0f} cm")

## 6. Cámaras en movimiento y multicámara

**Cámara de TV (paneo, tilt, zoom)** → la homografía cambia en cada frame. Opciones:
1. **Registro del campo por frame**: un modelo detecta keypoints/líneas del campo y se calcula H cada frame
   (SoccerNet *Camera Calibration*, PnLCalib, TVCalib). Robusto pero necesita ver suficientes líneas.
2. **Propagación**: H del frame t a partir del anterior y la homografía entre frames consecutivos
   (ORB/SIFT + RANSAC en el fondo, excluyendo jugadores): $H_{t} = H_{t-1} \cdot H_{t \to t-1}$. Acumula deriva
   → re-anclar periódicamente con (1).
3. Suavizar H en el tiempo (los parámetros de cámara cambian suavemente).

**Varias cámaras fijas**:
- Cada cámara $c$ tiene su $H_c$ hacia el **mismo plano del campo** → todas las detecciones se expresan en
  metros en un sistema común.
- **Asociación entre cámaras**: detecciones de cámaras distintas a < ~1 m en el campo (y apariencia compatible)
  son la misma persona → húngaro sobre distancias en el plano. Luego un **único tracker en coordenadas del campo**
  (Kalman en metros) → resuelve oclusiones de una cámara con otra.
- **Sincronización temporal** imprescindible (timestamps, genlock, o sincronizar por un evento visual/sonoro).
- **3D** (el balón en el aire, altura de un salto): no está en el plano → **triangulación** con ≥ 2 cámaras
  calibradas (K, R, t conocidas): `cv2.triangulatePoints(P1, P2, x1, x2)` con $P = K[R \mid t]$.
  La geometría epipolar (matriz fundamental $F$, $x_2^\top F x_1 = 0$) restringe dónde puede estar un punto de
  una cámara en la otra (una línea) → ayuda a asociar entre cámaras.

In [ ]:
# Mini demo de triangulación: un balón a 3 m de altura visto por la cámara de antes y otra en el fondo
ball = np.array([[60.0, 30.0, 3.0]])
cam2_pos = np.array([-25.0, 34.0, 12.0]); z2 = np.array([52.5, 34, 0]) - cam2_pos; z2 /= np.linalg.norm(z2)
x2 = np.cross(z2, [0, 0, 1.0]); x2 /= np.linalg.norm(x2); y2 = np.cross(z2, x2)
R2 = np.stack([x2, y2, z2]); t2 = -R2 @ cam2_pos
P1, P2 = Kc @ np.c_[Rc, tc], Kc @ np.c_[R2, t2]
u1, u2 = project(ball, Kc, Rc, tc), project(ball, Kc, R2, t2)
Xh = cv2.triangulatePoints(P1, P2, u1.T, u2.T); X = (Xh[:3] / Xh[3]).ravel()
print("balón real:", ball[0], "| triangulado:", X)
print("proyectarlo con la homografía del suelo daría:", apply_h(H_inv_true, u1)[0], "← ¡error de metros!")